<a href="https://colab.research.google.com/github/7vyshnav07-bot/Project/blob/main/Travel_PLanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **AI TRAVEL PLANNER**

### installing libraries

In [ ]:
!pip install langgraph

!pip install langchain

!pip install langchain-community

!pip install langchain-groq

!pip install langchain-text-splitters

!pip install faiss-cpu

!pip install sentence-transformers

!pip install duckduckgo-search

!pip install gradio

!pip install pypdf

!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 1.6 MB/s 

### Connect API

In [ ]:
from getpass import getpass
import os
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = getpass("Enter Groq API Key: ")

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile"
)

### Creating Folder for RAG

In [ ]:
import os

os.makedirs("data", exist_ok=True)

print("data folder created")

In [ ]:
import os

from langchain_community.document_loaders import (
    TextLoader
)

def load_all_documents():

    documents = []

    data_folder = "data"

    for file in os.listdir(data_folder):

        if file.endswith(".txt"):

            file_path = os.path.join(
                data_folder,
                file
            )

            loader = TextLoader(file_path)

            documents.extend(loader.load())

            print(f"Loaded TXT: {file}")

    return documents

In [ ]:
documents = load_all_documents()

print(len(documents))

In [ ]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

def split_documents(documents):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(documents)

    return chunks

In [ ]:
chunks = split_documents(documents)

print(len(chunks))

In [ ]:
from langchain_community.embeddings import (
    HuggingFaceEmbeddings
)

embeddings = HuggingFaceEmbeddings(
    model_name=
    "sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_community.vectorstores import (
    FAISS
)

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS DB Created")

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 8}
)


### Weather API

In [ ]:
import requests

def get_weather(latitude, longitude):

    try:

        url = (
            f"https://api.open-meteo.com/v1/forecast"
            f"?latitude={latitude}"
            f"&longitude={longitude}"
            f"&current_weather=true"
        )

        response = requests.get(url)

        data = response.json()

        return data["current_weather"]

    except Exception:

        return "Weather unavailable"

In [ ]:
district_data = {

    "thiruvananthapuram": {
        "coordinates": (8.5241, 76.9366),
        "famous_spots": [
            "Kovalam Beach",
            "Padmanabhaswamy Temple",
            "Varkala",
            "Poovar Island"
        ],
        "best_time": "October to March",
        "budget_range": "₹2500 - ₹7000/day",
        "trip_type": [
            "beach",
            "spiritual",
            "family"
        ]
    },

    "kollam": {
        "coordinates": (8.8932, 76.6141),
        "famous_spots": [
            "Thenmala",
            "Jatayu Earth Center",
            "Ashtamudi Lake",
            "Palaruvi Falls"
        ],
        "best_time": "September to February",
        "budget_range": "₹2000 - ₹5000/day",
        "trip_type": [
            "nature",
            "adventure",
            "eco tourism"
        ]
    },

    "pathanamthitta": {
        "coordinates": (9.2648, 76.7870),
        "famous_spots": [
            "Sabarimala",
            "Gavi",
            "Perunthenaruvi Falls",
            "Konni Forest"
        ],
        "best_time": "November to February",
        "budget_range": "₹2000 - ₹4500/day",
        "trip_type": [
            "pilgrimage",
            "forest",
            "peaceful"
        ]
    },

    "alappuzha": {
        "coordinates": (9.4981, 76.3388),
        "famous_spots": [
            "Alleppey Backwaters",
            "Houseboats",
            "Marari Beach",
            "Kuttanad"
        ],
        "best_time": "October to February",
        "budget_range": "₹3000 - ₹9000/day",
        "trip_type": [
            "backwaters",
            "honeymoon",
            "relaxation"
        ]
    },

    "kottayam": {
        "coordinates": (9.5916, 76.5222),
        "famous_spots": [
            "Kumarakom",
            "Illikkal Kallu",
            "Vagamon",
            "Marmala Waterfalls"
        ],
        "best_time": "September to March",
        "budget_range": "₹2500 - ₹6000/day",
        "trip_type": [
            "hill station",
            "nature",
            "peaceful"
        ]
    },

    "idukki": {
        "coordinates": (9.8490, 76.9720),
        "famous_spots": [
            "Munnar",
            "Thekkady",
            "Ramakkalmedu",
            "Kolukkumalai"
        ],
        "best_time": "September to March",
        "budget_range": "₹3000 - ₹8000/day",
        "trip_type": [
            "hill station",
            "adventure",
            "honeymoon"
        ]
    },

    "ernakulam": {
        "coordinates": (9.9816, 76.2999),
        "famous_spots": [
            "Fort Kochi",
            "Marine Drive",
            "Cherai Beach",
            "Mattancherry"
        ],
        "best_time": "October to February",
        "budget_range": "₹2500 - ₹7500/day",
        "trip_type": [
            "city",
            "heritage",
            "food"
        ]
    },

    "thrissur": {
        "coordinates": (10.5276, 76.2144),
        "famous_spots": [
            "Athirappilly Falls",
            "Thrissur Pooram",
            "Guruvayur Temple",
            "Snehatheeram Beach"
        ],
        "best_time": "October to February",
        "budget_range": "₹2000 - ₹6000/day",
        "trip_type": [
            "culture",
            "waterfalls",
            "family"
        ]
    },

    "palakkad": {
        "coordinates": (10.7867, 76.6548),
        "famous_spots": [
            "Silent Valley",
            "Malampuzha Dam",
            "Nelliyampathy",
            "Palakkad Fort"
        ],
        "best_time": "September to February",
        "budget_range": "₹2000 - ₹5000/day",
        "trip_type": [
            "nature",
            "trekking",
            "peaceful"
        ]
    },

    "malappuram": {
        "coordinates": (11.0732, 76.0740),
        "famous_spots": [
            "Kottakkunnu",
            "Nilambur",
            "Adyanpara Falls",
            "Teak Museum"
        ],
        "best_time": "October to February",
        "budget_range": "₹1800 - ₹4500/day",
        "trip_type": [
            "nature",
            "waterfalls",
            "budget"
        ]
    },

    "kozhikode": {
        "coordinates": (11.2588, 75.7804),
        "famous_spots": [
            "Kappad Beach",
            "Beypore",
            "SM Street",
            "Kozhikode Beach"
        ],
        "best_time": "October to February",
        "budget_range": "₹2500 - ₹6500/day",
        "trip_type": [
            "food",
            "beach",
            "city"
        ]
    },

    "wayanad": {
        "coordinates": (11.6854, 76.1320),
        "famous_spots": [
            "Edakkal Caves",
            "Banasura Sagar Dam",
            "Chembra Peak",
            "Soochipara Falls"
        ],
        "best_time": "October to May",
        "budget_range": "₹2500 - ₹7000/day",
        "trip_type": [
            "hill station",
            "trekking",
            "peaceful"
        ]
    },

    "kannur": {
        "coordinates": (11.8745, 75.3704),
        "famous_spots": [
            "Muzhappilangad Beach",
            "St. Angelo Fort",
            "Parassinikadavu",
            "Payyambalam Beach"
        ],
        "best_time": "October to March",
        "budget_range": "₹2000 - ₹5500/day",
        "trip_type": [
            "beach",
            "culture",
            "relaxation"
        ]
    },

    "kasaragod": {
        "coordinates": (12.4996, 74.9869),
        "famous_spots": [
            "Bekal Fort",
            "Ranipuram",
            "Ananthapura Lake Temple",
            "Kappil Beach"
        ],
        "best_time": "October to February",
        "budget_range": "₹2000 - ₹5000/day",
        "trip_type": [
            "forts",
            "nature",
            "peaceful"
        ]
    }

}

In [ ]:
district_aliases = {

    # Thiruvananthapuram
    "trivandrum": "thiruvananthapuram",
    "tvm": "thiruvananthapuram",

    # Kozhikode
    "calicut": "kozhikode",

    # Ernakulam
    "cochin": "ernakulam",
    "kochi": "ernakulam",

    # Alappuzha
    "alleppey": "alappuzha",

    # Kasaragod
    "kasargod": "kasaragod",

    # Idukki
    "munnar": "idukki",
    "thekkady": "idukki",

    # Wayanad
    "sultan bathery": "wayanad",

    # Kannur
    "cannanore": "kannur",

    # Thrissur
    "athirappilly": "thrissur",

    # Kottayam
    "vagamon": "kottayam",

    # Kollam
    "thenmala": "kollam"
}

In [ ]:
def planner_agent(user_query):

    query = user_query.lower()

    result = {

        "trip_type": None,
        "budget_type": None,
        "destination": None
    }

    # Trip type detection
    if "peaceful" in query:
        result["trip_type"] = "peaceful"

    # Budget detection
    if "budget" in query:
        result["budget_type"] = "budget"

    # Check aliases first
    for alias, real_name in district_aliases.items():

        if alias in query:

            result["destination"] = real_name

    # Check original district names
    for district in district_data.keys():

        if district in query:

            result["destination"] = district

    return result

In [ ]:
def classify_query(user_query):

    query = user_query.lower()

    itinerary_keywords = [

        "plan",
        "trip",
        "itinerary",
        "days",
        "travel",
        "vacation",
        "honeymoon",
        "visit",
        "tour",
        "weekend",
        "road trip"
    ]

    for word in itinerary_keywords:

        if word in query:

            return "itinerary"

    return "general"

In [ ]:
def kerala_travel_planner(user_query):

    # -------------------------
    # Query Type
    # -------------------------

    query_type = classify_query(
        user_query
    )

    # -------------------------
    # Retrieve Kerala Context
    # -------------------------

    docs = retriever.invoke(
        user_query
    )

    context = "\n".join(
        [
            doc.page_content
            for doc in docs
        ]
    )

    # -------------------------
    # Extract Source Files
    # -------------------------

    sources = list(set(

        [
            doc.metadata["source"].split("/")[-1]
            for doc in docs
        ]

    ))

    # -------------------------
    # Extract Query Info
    # -------------------------

    agent_result = planner_agent(
        user_query
    )

    destination = agent_result[
        "destination"
    ]

    # -------------------------
    # Weather
    # -------------------------

    weather = "Unavailable"

    try:

        if destination:

            lat, lon = district_data[
                destination
            ]["coordinates"]

            weather = get_weather(
                lat,
                lon
            )

    except:

        weather = "Unavailable"

    # -------------------------
    # Prompt
    # -------------------------

    prompt = f"""
    You are an expert Kerala travel assistant.

    Use the provided Kerala travel context
    and current weather information.

    Answer ONLY based on:
    - Kerala tourism
    - travel advice
    - retrieved context
    - weather information

    Kerala Context:
    {context}

    Current Weather:
    {weather}

    User Request:
    {user_query}

    If the user asks for:
    - itinerary → provide day-wise plan
    - travel tips → answer directly
    - weather → explain weather
    - destination suggestions → recommend places

    Keep answers natural,
    concise,
    and helpful.
    """

    # -------------------------
    # LLM Response
    # -------------------------

    response = llm.invoke(
        prompt
    )

    # -------------------------
    # Format Sources
    # -------------------------

    source_text = "\n".join(

        [
            f"• {source}"
            for source in sources
        ]

    )

    # -------------------------
    # Final Response
    # -------------------------

    final_response = f"""

{response.content}

📚 Sources Used:
{source_text}
"""

    return final_response

Web Hosting


In [ ]:
import gradio as gr

# -------------------------
# Response Function
# -------------------------

def respond(message, history):

    try:

        bot_response = kerala_travel_planner(
            message
        )

    except Exception as e:

        bot_response = (
            f"❌ Error: {str(e)}"
        )

    history.append(
        {
            "role": "user",
            "content": message
        }
    )

    history.append(
        {
            "role": "assistant",
            "content": bot_response
        }
    )

    return "", history

# -------------------------
# Custom CSS
# -------------------------

custom_css = """

body {
    background:
    linear-gradient(
        135deg,
        #081c15,
        #1b4332
    );
}

/* Main container */
.gradio-container {
    max-width: 1200px !important;
    margin: auto;
}

/* Hide footer */
footer {
    visibility: hidden;
}

/* Title */
#title {
    text-align: center;
    font-size: 46px;
    font-weight: 700;
    color: white;
    margin-top: 10px;
}

#subtitle {
    text-align: center;
    color: #d8f3dc;
    margin-bottom: 25px;
    font-size: 18px;
}

/* Chatbot */
.chatbot {
    border-radius: 22px !important;
    background: rgba(255,255,255,0.07) !important;
    backdrop-filter: blur(12px);
    border: 1px solid rgba(255,255,255,0.08);
    padding: 10px;
}

/* Textbox */
textarea {
    border-radius: 18px !important;
    background: rgba(255,255,255,0.08) !important;
    color: white !important;
    border: 1px solid rgba(255,255,255,0.08) !important;
    padding: 16px !important;
    font-size: 16px !important;
    min-height: 70px !important;
}

/* Placeholder */
textarea::placeholder {
    color: #d8f3dc !important;
}

/* Buttons */
button {
    border-radius: 14px !important;
}

/* Sidebar */
.sidebar {
    background: rgba(255,255,255,0.05);
    border-radius: 22px;
    padding: 18px;
    border: 1px solid rgba(255,255,255,0.08);
}

"""

# -------------------------
# Frontend
# -------------------------

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=custom_css
) as demo:

    gr.Markdown(
        """
        <div id="title">
            🌴 Kerala Travel AI
        </div>

        <div id="subtitle">
            Explore God's Own Country with AI
        </div>
        """
    )

    with gr.Row():

        # Sidebar
        with gr.Column(
            scale=1,
            elem_classes="sidebar"
        ):

            gr.Markdown(
                """
                ### 🌟 Features

                ✅ Smart Trip Planning
                ✅ Live Weather
                ✅ Kerala Tourism Guide
                ✅ Budget Travel Suggestions
                """
            )

            clear = gr.Button(
                "🗑️ Clear Chat"
            )

        # Main Chat Area
        with gr.Column(scale=4):

            chatbot = gr.Chatbot(
                type="messages",
                height=600,
                elem_classes="chatbot",
                bubble_full_width=False
            )

            with gr.Row(equal_height=True):

                msg = gr.Textbox(
                   placeholder=
                   "Ask anything about Kerala tourism...",
                   lines=1,
                    max_lines=1,
                   show_label=False,
                   scale=8
            )
                submit = gr.Button(
                    "➤",
                    variant="primary",
                    min_width=70
                )

    # Submit by Enter
    msg.submit(
        respond,
        [msg, chatbot],
        [msg, chatbot]
    )

    # Submit by Button
    submit.click(
        respond,
        [msg, chatbot],
        [msg, chatbot]
    )

    # Clear Chat
    clear.click(
        lambda: [],
        None,
        chatbot,
        queue=False
    )

# -------------------------
# Launch
# -------------------------

demo.launch(
    share=True,
    debug=True
)